In [1]:
from test.test_levy_area import conditional_statistics

import jax
import jax.numpy as jnp
import jax.random as jrandom
from diffrax.brownian.tree import VirtualBrownianTree
from jax import config


config.update("jax_enable_x64", True)
%env JAX_PLATFORM_NAME=cuda


key = jrandom.PRNGKey(0)

env: JAX_PLATFORM_NAME=cuda


In [2]:
ts = jnp.linspace(0.3, 2.3, 10000)


def vec_eval(_tree, _ts):
    return jax.vmap(lambda _t: _tree.evaluate(_t, use_levy=True))(_ts)

In [3]:
tree_deep = VirtualBrownianTree(
    t0=0.3, t1=2.3, tol=2**-17, shape=(10,), key=key, levy_area="space-time-time"
)
tree_shallow = VirtualBrownianTree(
    t0=0.3, t1=2.3, tol=2**-5, shape=(10,), key=key, levy_area="space-time-time"
)
vals = tree_deep.evaluate(0.56172, 0.772984, use_levy=True)
print(f"r {vals.dt} W {vals.W}, H {vals.H}, K {vals.K}")
print(vals)

r 0.21126399999999998 W [-0.54120149 -0.00424841 -0.54207085 -0.76960388  0.91057222  0.27000034
 -0.22443617 -0.66879882 -0.25051934  0.60908545], H [ 0.08242439  0.15235178 -0.21342129 -0.06026207 -0.00636434 -0.0241145
  0.14789518 -0.33223547 -0.13661979 -0.06587297], K [ 0.00146567  0.01727792 -0.00587919  0.00635263  0.01244853  0.01569882
 -0.00321116  0.01891226  0.00796841  0.00835777]
LevyVal(dt=f64[], W=f64[10], H=f64[10], bar_H=None, K=f64[10], bar_K=None)


# Timeing
## First without final interpolation

In [4]:
jax.block_until_ready(vec_eval(tree_deep, ts))
%timeit jax.block_until_ready(vec_eval(tree_deep, ts))

22.3 ms ± 241 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
jax.block_until_ready(vec_eval(tree_shallow, ts))
%timeit jax.block_until_ready(vec_eval(tree_shallow, ts))

9.19 ms ± 396 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [6]:
conditional_statistics("space-time-time")  # without final interpolation (tol=2**-12)

total_mean_err 1.089e-03, total_cov_err 2.102e-04


In [7]:
conditional_statistics("space-time-time", 2**-8)  # without final interpolation

total_mean_err 1.084e-03, total_cov_err 2.139e-04


## Now with final interpolation

In [4]:
jax.block_until_ready(vec_eval(tree_deep, ts))
%timeit jax.block_until_ready(vec_eval(tree_deep, ts))

28.2 ms ± 69.7 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
jax.block_until_ready(vec_eval(tree_shallow, ts))
%timeit jax.block_until_ready(vec_eval(tree_shallow, ts))

15.8 ms ± 236 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [4]:
conditional_statistics("space-time-time")  # with final interpolation (tol=2**-12)

total_mean_err 1.101e-03, total_cov_err 2.260e-04


In [5]:
conditional_statistics("space-time-time", 2**-8)  # with final interpolation

total_mean_err 1.440e-03, total_cov_err 1.388e-02
